Week 4 – ETL in Azure Databricks

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# 1. Load Cleaned Data 
df_movements = spark.read.option("header", "true").option("inferSchema", "true").csv("/Workspace/Users/jstevemanuel@gmail.com/stock_movements.csv")

In [0]:
# 2. Process Aggregate Stock
stock_agg = df_movements.groupBy("product_id", "warehouse_id").agg(
    F.sum("quantity").alias("current_stock"),
    F.first("product_name").alias("product_name"),
    F.first("reorder_level").alias("reorder_level")
)

In [0]:
# 3. Create Master Inventory View with Reorder Flag (Requirement)
master_inventory_view = stock_agg.withColumn(
    "needs_reorder",
    F.when(F.col("current_stock") < F.col("reorder_level"), "YES")
     .otherwise("NO")
)

In [0]:
# 4. Save Output in Delta Format
output_path = "gold_inventory_master"

master_inventory_view.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(output_path)

print("Master Inventory View Generated:")
master_inventory_view.show()

Master Inventory View Generated:
+----------+------------+-------------+--------------------+-------------+-------------+
|product_id|warehouse_id|current_stock|        product_name|reorder_level|needs_reorder|
+----------+------------+-------------+--------------------+-------------+-------------+
|       412|   WH-MUM-01|            7|           USB-C Hub|           15|          YES|
|       205|   WH-BLR-02|          160|      Wireless Mouse|           20|           NO|
|       101|   WH-MUM-01|            7|High-Performance ...|           10|          YES|
|       309|   WH-DEL-03|            3| Mechanical Keyboard|            5|          YES|
+----------+------------+-------------+--------------------+-------------+-------------+



Week 5 – Pipeline Automation with Azure
DevOps

In [0]:
import pandas as pd
import numpy as np
from datetime import datetime

def generate_daily_report():
    # 1. Load Data
    df = pd.read_csv("/Workspace/Users/jstevemanuel@gmail.com/stock_movements.csv")

    # 2. Aggregate Stock 
    stock_summary = df.groupby(['warehouse_id', 'product_id', 'product_name']).agg({
        'quantity': 'sum',
        'reorder_level': 'first'
    }).reset_index()

    # 3. Identify Low-Stock Items
    reorder_list = stock_summary[stock_summary['quantity'] < stock_summary['reorder_level']]

    # 4. Export CSV Deliverable
    filename = "reorder_list.csv"
    reorder_list.to_csv(filename, index=False)
    
    print(f"Report Generated: {len(reorder_list)} items need reordering.")
    print(reorder_list)
generate_daily_report()


Report Generated: 3 items need reordering.
  warehouse_id  product_id             product_name  quantity  reorder_level
1    WH-DEL-03         309      Mechanical Keyboard         3              5
2    WH-MUM-01         101  High-Performance Laptop         7             10
3    WH-MUM-01         412                USB-C Hub         7             15
